## Fan Engagement & Content Performance of SM Entertainment Artists

**Scope & Limitations**
* Data: official YouTube channels of 7 groups, uploads from 2025-01-01 to the collection date (YouTube Data API v3). TikTok and Instagram are not covered.
* Video type is derived from duration (<= 180 s = "Shorts"), since the API has no Shorts flag. About 96% of labelled Shorts are <= 60 s.
* Main releases include single albums, mini albums, studio albums, and selected digital singles (full promotion with MV).
* Results are descriptive patterns. Videos cluster around 28 releases, so they do not show causation.

### Libraries

In [153]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import kruskal

ROOT = Path.cwd().parent          
sys.path.append(str(ROOT))

from config import PROCESSED
from config import PRE_DAYS, DURING_DAYS, POST_DAYS

### Load Data


In [136]:
df_yt = pd.read_csv(PROCESSED / "youtube_clean.csv", parse_dates=["published_at", "collected_at"])
df_rel = pd.read_csv(PROCESSED / "releases_clean.csv", parse_dates=["release_date"])

print(df_yt.shape)
print(df_rel.columns.tolist())
print(df_yt["group"].unique() == df_rel["group"].unique())   


(6164, 12)
['release_id', 'group', 'release_name', 'release_date', 'type', 'is_main_releases']
[ True  True  True  True  True  True  True]


### 1. Feature Engineering
Columns added to make videos comparable across groups, formats, and upload dates:
- **Performance:** `age_days`, `views_per_day`, `engagement_pct`, `relative_vpd`, `relative_engagement`
- **Posting time (KST):** `hour_kst`, `day_kst`, `month`
- **Comeback:** `phase`, `days_from_release`, `release_id`

Engagement is (likes + comments) / views. If comments are disabled, likes only.
`relative_*` columns are divided by the median of the same group and video type, so 1.0 is a typical video.

Phase is based on the nearest main release (R):
- pre: 30 days before R
- during: R to R + 14 days
- post: R + 15 to R + 56 days
- break: the rest

Videos under **7 days old** are **excluded** because their views are still unstable.

In [137]:
yt = df_yt.copy()
rel = df_rel.copy()

# Video Age (in days) at the Time the Data was Collected
yt["age_days"] = (yt["collected_at"] - yt["published_at"]).dt.days.clip(lower=1)
yt["views_per_day"] = yt["views"] / yt["age_days"]
# print(yt["views_per_day"].round(2))

# Engagement Rate (%); Views = 0 Becomes NaN to Avoid Division by Zero
views = yt["views"].replace(0, np.nan)
yt["engagement_pct"] = np.where(
    yt["comments_disabled"],
    yt["likes"] / views * 100, # Comments Disabled: Likes Only
    (yt["likes"] + yt["comments"]) / views * 100,
)
# print(yt["engagement_pct"].round(2))

# Posting Time in KST (UTC+9)
kst = yt["published_at"] + pd.Timedelta(hours=9)
yt["hour_kst"] = kst.dt.hour
yt["day_kst"] = kst.dt.day_name()
yt["month"] = kst.dt.to_period("M")
yt["year"] = kst.dt.year
# print(yt[["hour_kst", "day_kst", "month"]])

# Exclude Very Recent Videos (View Counts are Still Unstable)
yt_eval = yt[yt["age_days"] >= 7].copy()
# print(len(yt), "->", len(yt_eval))

# Relative performance: 1.0 = typical video for that group and video type
grp = yt_eval.groupby(["group", "video_type"])
yt_eval["relative_vpd"] = yt_eval["views_per_day"] / grp["views_per_day"].transform("median")
yt_eval["relative_engagement"] = yt_eval["engagement_pct"] / grp["engagement_pct"].transform("median")

# Main releases only, sorted by date
rel_main = rel[rel["is_main_releases"]].sort_values("release_date")

def assign_phase(group, date):
    """Label a video by its position relative to the group's nearest main release."""
    dates = rel_main.loc[rel_main["group"] == group, "release_date"]
    if dates.empty:
        return pd.Series(["break", np.nan, np.nan])
    diff = (date.normalize() - dates).dt.days          
    nearest = diff.abs().idxmin()
    d = diff.loc[nearest]
    if -PRE_DAYS <= d < 0:
        phase = "pre_comeback"
    elif 0 <= d <= DURING_DAYS:
        phase = "during_comeback"
    elif DURING_DAYS < d <= DURING_DAYS + POST_DAYS:
        phase = "post_comeback"
    else:
        phase = "break"
    return pd.Series([phase, d, rel_main.loc[nearest, "release_id"]])

yt_eval[["phase", "days_from_release", "release_id"]] = yt_eval.apply(
    lambda r: assign_phase(r["group"], r["published_at"] + pd.Timedelta(hours=9)), axis=1
)

### 2. Data Overview & Distribution
**Findings**
* 6,164 videos from 7 channels (2025-01-06 to 2026-11-16); 6,086 kept after excluding videos under 7 days old.
* Shorts make up 78.6% of uploads. H2H has the highest share (85.4%), while NCT 127 has the lowest (66.9%).
* Shorts share rises from 71.3% in `break` to 85.3% in `during_comeback`, showing a heavy short-form push during active promotions.
* Views per day is strongly right-skewed (mean is 3-4x the median; views 2-3x), so medians and relative ratios are used for comparisons.
* Shorts reach more viewers (median views per day 1,349 vs 1,019 for Long), but Long videos get about 3x more comments per video (median 717 vs 242).
* Engagement rate is higher for Shorts (median 6.66% vs 5.74%), mostly driven by likes.

**Notes**:
- NCT 127 has only 1 main release and WayV has 2, so group-level results for them are weak.
- 2026 is a partial year and its videos are younger, so cross-period comparisons use views per day instead of total views.


In [138]:
print("="*45)
print("--------------- DATA COVERAGE --------------- ")
print("="*45)
print("Total Videos (Initial):", len(yt))
print("Total Videos Evaluated (age >= 7d):", len(yt_eval), "\nVideo that is not included: ", len(yt) - len(yt_eval))
print("Video Date Range:", yt_eval["published_at"].min().date(), "to", yt_eval["published_at"].max().date())
print("Total Group to Analyze:", yt_eval['group'].nunique())
print("Total Main Releases:", len(rel_main))
print(rel_main.groupby("group").size())


--------------- DATA COVERAGE --------------- 
Total Videos (Initial): 6164
Total Videos Evaluated (age >= 7d): 6086 
Video that is not included:  78
Video Date Range: 2025-01-06 to 2026-11-16
Total Group to Analyze: 7
Total Main Releases: 28
group
aespa            5
hearts2hearts    7
nct 127          1
nct dream        3
nct wish         6
riize            4
wayv             2
dtype: int64


In [139]:
print("\n"+"="*45)
print("--------- CATEGORICAL DISTRIBUTION ----------")
print("="*45)
print("\n--- Distribution by Group and Video Type ---")
dist_group_type = pd.crosstab(
    yt_eval["group"], yt_eval["video_type"], 
    margins=True, 
    margins_name="Total"
    )    
# Add Shorts % to show the share of short-form videos per group
dist_group_type["Shorts_%"] = (dist_group_type["Shorts"] / dist_group_type["Total"] * 100).round(1)
group_no_total = dist_group_type.drop("Total").sort_values(by="Shorts_%", ascending=False)
dist_group_type_sorted = pd.concat([group_no_total, dist_group_type.loc[["Total"]]])
print(dist_group_type_sorted)


print("\n------- Distribution by Comeback Phase ------- ")
dist_phase = pd.crosstab(
    yt_eval["phase"], yt_eval["video_type"], 
    margins=True, 
    margins_name="Total"
    )
dist_phase["Shorts_%"] = (dist_phase["Shorts"] / dist_phase["Total"] * 100).round(1)

phase_no_total = dist_phase.drop("Total").sort_values(by="Shorts_%", ascending=False)
dist_phase_sorted = pd.concat([phase_no_total, dist_phase.loc[["Total"]]])
print(dist_phase_sorted)




--------- CATEGORICAL DISTRIBUTION ----------

--- Distribution by Group and Video Type ---
video_type     Long  Shorts  Total  Shorts_%
group                                       
hearts2hearts   209    1222   1431     85.40
riize           277    1132   1409     80.30
nct wish        275    1021   1296     78.80
aespa           202     609    811     75.10
wayv            111     329    440     74.80
nct dream       134     287    421     68.20
nct 127          92     186    278     66.90
Total          1300    4786   6086     78.60

------- Distribution by Comeback Phase ------- 
video_type       Long  Shorts  Total  Shorts_%
phase                                         
during_comeback   242    1401   1643     85.30
pre_comeback      230     964   1194     80.70
post_comeback     293    1091   1384     78.80
break             535    1330   1865     71.30
Total            1300    4786   6086     78.60


In [140]:
print("\n"+"="*50)
print("------------- NUMERICAL DISTRIBUTION ------------- ")
print("="*50)

# Format float
pd.options.display.float_format = '{:,.2f}'.format

metrics = ["duration", "views", "likes", "comments", "views_per_day", "engagement_pct", "relative_vpd"]

print(yt_eval.groupby("video_type")[metrics].describe().T.drop(["count", "std"], level=1))


------------- NUMERICAL DISTRIBUTION ------------- 
video_type                   Long         Shorts
duration       mean      1,439.77          22.07
               min         181.00           3.00
               25%         804.50          12.00
               50%       1,423.00          16.00
               75%       1,938.00          23.00
               max       7,423.00         179.00
views          mean    519,328.21     940,454.63
               min       5,221.00       6,455.00
               25%     142,029.75     142,164.25
               50%     261,411.00     319,685.00
               75%     542,748.00     700,714.25
               max  18,486,614.00 167,860,560.00
likes          mean     21,741.41      35,619.29
               min         636.00       1,052.00
               25%       8,384.50      10,910.75
               50%      14,198.50      20,289.00
               75%      25,062.25      39,718.00
               max     377,093.00   2,326,331.00
comments       m

### 3. Comeback Effect

**Findings**
* **Long-form videos peak around a release.** Median relative VPD is ~1.3x baseline in `pre_comeback` (1.34x raw, 1.29x age-adjusted) and `during_comeback` (1.31x, 1.26x), then drops to ~0.9x in `post_comeback`. The pattern holds after age adjustment.
* **Shorts show little phase effect once video age is controlled.** Raw `during_comeback` looks weak (0.94x vs 1.10x in `break`), but age-adjusted values are ~1.0x in `during` (1.05x), `post` (1.04x) and `break` (1.01x). Only `pre_comeback` lags (0.86x).
* **Engagement stays flat across phases** (0.97x to 1.06x for both formats): comebacks affect reach, not interaction per viewer.
* **Group-level patterns differ by format.** For Long videos, RIIZE (1.99x) and NCT Wish (1.46x) peak in `pre_comeback`, while aespa peaks in `during_comeback` (1.75x vs 0.54x in `pre`). For Shorts, aespa (1.49x) and H2H (1.45x) are highest in `break`, and NCT Dream in `pre_comeback` (1.44x).

**Implication for the marketing team**
* Schedule key Long-form content (MVs, dance practices) in the weeks before and during a release.
* Shorts performance is steady across phases, so they can fill the calendar all year.

**Notes**
* 9 group-phase cells with fewer than 30 videos are marked `*` and excluded from interpretation.
* Group-level tables use raw relative VPD (not age-adjusted).
* Videos cluster around 28 main releases rather than 6,086 independent events, so Kruskal-Wallis results (p < 0.001) are directional, not causal proof.
* Phase windows (30 / 14 / 56 days) and the inclusion of digital singles as main releases are assumptions; no sensitivity check was run.

In [141]:
# Chronological order of comeback phases
phase_order = ["pre_comeback", "during_comeback", "post_comeback", "break"]
yt_eval["phase"] = pd.Categorical(yt_eval["phase"], categories=phase_order, ordered=True)

# Median performance per phase (1.0 = typical video of that group and video type)
phase_summary = (
    yt_eval.groupby(["video_type", "phase"], observed=False)
    .agg(
        n_videos=("video_id", "count"),
        median_rel_vpd=("relative_vpd", "median"),
        median_rel_eng=("relative_engagement", "median"),
    )
    .round(2)
)
print("\n"+"="*70)
print("------------------- AGGREGATE PERFORMANCE BY PHASE  ------------------ ")
print("="*70)
print(phase_summary)


------------------- AGGREGATE PERFORMANCE BY PHASE  ------------------ 
                            n_videos  median_rel_vpd  median_rel_eng
video_type phase                                                    
Long       pre_comeback          230            1.34            1.01
           during_comeback       242            1.31            1.00
           post_comeback         293            0.85            1.02
           break                 535            0.96            0.98
Shorts     pre_comeback          964            0.83            1.06
           during_comeback      1401            0.94            0.97
           post_comeback        1091            1.08            0.97
           break                1330            1.10            1.01


In [142]:
# Find Correlation (Video Age vs Views Per Day)
spearman_r = yt_eval["age_days"].corr(yt_eval["views_per_day"], method="spearman")
print(f"Spearman (age_days vs views_per_day): {spearman_r:.2f}")

# Age-adjusted relative VPD: compare videos only within the same age quartile
yt_eval["age_band"] = pd.qcut(
    yt_eval["age_days"], q=4, labels=["youngest", "young", "old", "oldest"]
)
grp_age = yt_eval.groupby(["group", "video_type", "age_band"], observed=True)
yt_eval["rel_vpd_age_adj"] = (
    yt_eval["views_per_day"] / grp_age["views_per_day"].transform("median")
)

age_adj = (
    yt_eval.groupby(["video_type", "phase"], observed=False)["rel_vpd_age_adj"]
    .median().round(2).unstack("video_type")
)

# Compare Raw vs age-adjusted, side by side
compare = (
    phase_summary["median_rel_vpd"].unstack("video_type")
    .join(age_adj, lsuffix="_raw", rsuffix="_age_adj")
)
print("\n"+"="*70)
print("------------------ RAW vs AGE-ADJUSTED RELATIVE VPD ------------------ ")
print("="*70)
print(compare)

# Calculate Kruskal-Wallis on both versions (Raw VS Adjusted)
print("\n"+"="*50)
print("----------------- KRUSKAL-WALLIS ----------------- ")
print("="*50)

for col in ["relative_vpd", "rel_vpd_age_adj"]:
    for vtype in ["Shorts", "Long"]:
        sub = yt_eval[yt_eval["video_type"] == vtype]
        samples = [sub.loc[sub["phase"] == p, col].dropna() for p in phase_order]
        stat, p = kruskal(*samples)
        p_text = "p < 0.0001" if p < 0.0001 else f"p = {p:.4f}"
        print(f"{col} | {vtype}: H = {stat:.1f}, {p_text}")

Spearman (age_days vs views_per_day): -0.51

------------------ RAW vs AGE-ADJUSTED RELATIVE VPD ------------------ 
video_type       Long_raw  Shorts_raw  Long_age_adj  Shorts_age_adj
phase                                                              
pre_comeback         1.34        0.83          1.29            0.86
during_comeback      1.31        0.94          1.26            1.05
post_comeback        0.85        1.08          0.90            1.04
break                0.96        1.10          0.98            1.01

----------------- KRUSKAL-WALLIS ----------------- 


C:\Users\faust\AppData\Local\Temp\ipykernel_1588\2655395496.py:22: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  .join(age_adj, lsuffix="_raw", rsuffix="_age_adj")


relative_vpd | Shorts: H = 41.5, p < 0.0001
relative_vpd | Long: H = 18.0, p = 0.0004
rel_vpd_age_adj | Shorts: H = 47.2, p < 0.0001
rel_vpd_age_adj | Long: H = 26.2, p < 0.0001


In [143]:
# Sample size by group, video type, and phase
video_counts = yt_eval.pivot_table(
    index="group",
    columns=["video_type", "phase"],
    values="video_id",
    aggfunc="count",
    observed=False,
)

MIN_SAMPLE_SIZE = 30

small_sample_cells = (video_counts < MIN_SAMPLE_SIZE).sum().sum()

print(
    f"{small_sample_cells} cells have fewer than "
    f"{MIN_SAMPLE_SIZE} videos (marked with *)"
)

def group_phase_pivot(value_column):
    median_performance = yt_eval.pivot_table(
        index="group",
        columns=["video_type", "phase"],
        values=value_column,
        aggfunc="median",
        observed=False,
    ).round(2)

    formatted_values = median_performance.map(
        lambda x: f"{x:.2f}"
    )  

    return formatted_values.where(
        video_counts >= MIN_SAMPLE_SIZE,
        formatted_values + "*" # keep all values, mark with "*" for small sample cells
    )  

print("\n"+"="*75)
print("------------------- MEDIAN RELATIVE VPD BY GROUP & PHASE ------------------ ")
print("="*75)
print(group_phase_pivot("relative_vpd"))


print("\n"+"="*75)
print("---------------- MEDIAN RELATIVE ENGAGEMENT BY GROUP & PHASE -------------- ")
print("="*75)
print(group_phase_pivot("relative_engagement"))

print(
    f"\n* = fewer than {MIN_SAMPLE_SIZE} videos, "
    "interpret with caution. "
    "1.0 = typical video of that group and format."
)

9 cells have fewer than 30 videos (marked with *)

------------------- MEDIAN RELATIVE VPD BY GROUP & PHASE ------------------ 
video_type            Long                                           Shorts  \
phase         pre_comeback during_comeback post_comeback break pre_comeback   
group                                                                         
aespa                 0.54            1.75          0.84  0.99         0.69   
hearts2hearts         1.25            0.92          0.86  1.01         0.95   
nct 127              2.07*           1.44*         1.37*  0.68         0.72   
nct dream            1.81*            0.95         0.50*  1.08         1.44   
nct wish              1.46            1.44          0.72  0.99         0.70   
riize                 1.99            1.35          0.83  0.95         1.10   
wayv                 0.70*           1.11*         1.02*  0.99         0.54   

video_type                                         
phase         during_comeback

### 4. Posting Time

**Findings**
* **Upload timing is highly concentrated.** 94% of Long videos are posted 18-23 KST, so timing effects for Long cannot be assessed beyond day of week. Shorts are split between afternoon (43%) and evening (50%).
* **Shorts posted at night (00-05 KST) or in the morning (06-11 KST) underperform.** Median relative VPD is ~0.75 (age-adjusted), roughly 25% below baseline. Afternoon (1.00x) and evening (1.05x) are similar.
* **Day of week has a weak effect.** Long videos on Monday are the strongest (1.20x age-adjusted) and Shorts on Wednesday the weakest (0.88x). All other days fall within ~0.9x-1.1x. Monday's strength is not explained by comeback timing, since its share of pre/during videos is similar to other days.
* **Engagement is flat across days and time slots** (about 0.95x-1.05x in slots with enough data).

**Implication for the marketing team**
* Shorts: avoid uploading at night or in the morning. Anywhere between 12:00 and 23:59 KST performs similarly.
* Long: the data cannot support a timing recommendation because almost all uploads are already in the evening.

**Notes**
* Single-hour results are not interpreted. The 18:00 KST Long slot looks extreme (2.90x, n = 34), but its top 10 videos are mostly MVs and dance/performance videos, which suggests content type rather than the hour.
* Upload times are chosen by the channel, so the best days and hours may reflect the type of content posted then. Content type is not controlled in this analysis.


In [144]:
# Day of week in chronological order
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
yt_eval["day_kst"] = pd.Categorical(yt_eval["day_kst"], categories=day_order, ordered=True)

day_summary = (
    yt_eval.groupby(["video_type", "day_kst"], observed=False)
    .agg(
        n_videos=("video_id", "count"),
        median_rel_vpd=("relative_vpd", "median"),
        median_rel_vpd_adj=("rel_vpd_age_adj", "median"),
        median_rel_eng=("relative_engagement", "median"),
    )
    .round(2)
)
print("\n"+"="*65)
print("---------------- PERFORMANCE BY DAY OF WEEK (KST) --------------- ")
print("="*65)
print(day_summary)


---------------- PERFORMANCE BY DAY OF WEEK (KST) --------------- 
                      n_videos  median_rel_vpd  median_rel_vpd_adj  \
video_type day_kst                                                   
Long       Monday          221            1.21                1.20   
           Tuesday         202            1.03                1.03   
           Wednesday       216            0.89                0.99   
           Thursday        187            0.88                0.91   
           Friday          242            1.10                1.00   
           Saturday        114            0.80                0.95   
           Sunday          118            0.92                1.06   
Shorts     Monday          698            1.04                1.10   
           Tuesday         677            1.00                1.02   
           Wednesday       694            0.88                0.88   
           Thursday        654            1.00                1.01   
           Friday     

In [145]:
MIN_N = 30

# Hours grouped into time slots (KST)
bins = [0, 6, 12, 18, 24]
slot_labels = ["00-05 night", "06-11 morning", "12-17 afternoon", "18-23 evening"]
yt_eval["time_slot"] = pd.cut(yt_eval["hour_kst"], bins=bins, labels=slot_labels, right=False)

slot_summary = (
    yt_eval.groupby(["video_type", "time_slot"], observed=False)
    .agg(
        n_videos=("video_id", "count"),
        median_rel_vpd=("relative_vpd", "median"),
        median_rel_vpd_adj=("rel_vpd_age_adj", "median"),
        median_rel_eng=("relative_engagement", "median"),
    )
    .round(2)
)

print("\n"+"="*70)
print("------------------- PERFORMANCE BY TIME SLOT (KST) ------------------- ")
print("="*70)
print(slot_summary)

hour_summary = (
    yt_eval.groupby(["video_type", "hour_kst"], observed=False)
    .agg(
        n_videos=("video_id", "count"),
        median_rel_vpd_adj=("rel_vpd_age_adj", "median"),
        median_rel_eng=("relative_engagement", "median"),
    )
    .round(2)
)
print(f"\n=== HOURS WITH n >= {MIN_N} (sorted by age-adjusted VPD) ===")
for vtype in ["Shorts", "Long"]:
    sub = hour_summary.loc[vtype]
    valid = sub[sub["n_videos"] >= MIN_N].sort_values("median_rel_vpd_adj", ascending=False)
    print(f"\n--- {vtype} ---")
    print(valid)


------------------- PERFORMANCE BY TIME SLOT (KST) ------------------- 
                            n_videos  median_rel_vpd  median_rel_vpd_adj  \
video_type time_slot                                                       
Long       00-05 night            25            1.48                2.24   
           06-11 morning           3            0.51                0.73   
           12-17 afternoon        44            0.59                0.54   
           18-23 evening        1228            1.00                1.00   
Shorts     00-05 night           250            0.73                0.76   
           06-11 morning          86            0.80                0.76   
           12-17 afternoon      2062            1.00                1.00   
           18-23 evening        2388            1.05                1.05   

                            median_rel_eng  
video_type time_slot                        
Long       00-05 night                1.27  
           06-11 morning       

In [146]:
# Check 1: what is behind the extreme Long result at 18:00 KST?
print("---- Check the cause of the extreme result at 18:00 KST ---")
print(yt_eval[(yt_eval["video_type"] == "Long") & (yt_eval["hour_kst"] == 18)]
      .nlargest(10, "views_per_day")[["group", "title", "views", "views_per_day", "relative_vpd"]]) # Top 10 Videos

# Check 2: is Monday's strength just because comeback content?
print("\n--- Check monday's strength is it because comeback content? ---")
long_df = yt_eval[yt_eval["video_type"] == "Long"]
print(pd.crosstab(long_df["day_kst"], long_df["phase"], normalize="index").round(2))

---- Check the cause of the extreme result at 18:00 KST ---
              group                                              title  \
997   hearts2hearts                  Hearts2Hearts 하츠투하츠 'Moonride' MV   
1089  hearts2hearts  Hearts2Hearts 하츠투하츠 'ICONIC HEART' Dance Practice   
1522  hearts2hearts         Hearts2Hearts 하츠투하츠 'RUDE!' Dance Practice   
2208  hearts2hearts  Hearts2Hearts 하츠투하츠 'The Chase' Performance Video   
317           aespa                aespa 에스파 'LEMONADE' Dance Practice   
3280       nct wish  NCT WISH 엔시티 위시 'BOY MEETS GIRL' @NCTzen WISH-...   
669           aespa        aespa x PUBG 'Dark Arts' Collaboration Film   
3279       nct wish  NCT WISH 엔시티 위시 'YO-I-DON!' @NCTzen WISH-JAPAN...   
110           aespa    장담하는데 에스파 숙소가 세상에서 제일 시끄러움 🔊 | ❮aesParty❯ EP.15   
5552          riize  RIIZE 라이즈 '모든 하루의 끝 (The End of the Day)' MV |...   

         views  views_per_day  relative_vpd  
997    5571203      75,286.53         33.31  
1089   5200005      51,485.20    

### 5. Trend & Outliers

**Findings**
* **Raw trends are driven by video age.** Median views per day rises ~3-4x from 2025 to 2026, but age-adjusted values are flat (Long 0.98 → 1.02, Shorts 0.95 → 1.03), so there is no evidence that per-video performance improved.
* **Shorts upload volume grew, Long stayed flat.** Roughly 166 → 254 Shorts per month, while Long stayed at ~55-59 per month.
* **Views are concentrated.** The top 10 videos hold 17% of all views, the top 1% hold 30%, and the top 10% hold 61%. aespa is the most concentrated (69% from its top 10%), NCT Dream the least (38%).
* **Comebacks change the tail of Shorts, not the typical video.** Viral Shorts (relative VPD > 10) make up ~7% of Shorts uploaded during and after a comeback, vs 1.9% in pre and 3.1% in break, even though median Shorts performance barely changes. For Long videos the viral rate is similar across phases (4.5-7%).
* **Release benchmarks show no clear album vs digital-single difference.** After age adjustment, most releases fall within 0.8x-1.2x.
* **Reach and engagement move in opposite directions at the extremes.** Most top outliers have engagement of 0.3-1.5%, while most of the lowest-reach videos (tour recaps, countdown replays, teasers) have 8-20%. This may reflect a core-fan audience (hypothesis).
* **Several top outliers are YouTube Shorts-branded campaigns** (e.g. "Create a Short with Insomnia"). Whether these were paid or platform partnerships cannot be determined from the data.

**Notes**
* Raw VPD comparisons across time or releases are not valid because recent videos are younger. Age adjustment uses coarse quartiles, so the latest months (especially Nov 2026) and recent releases are not interpreted.
* Outliers are described, not explained: the data has no information on promotion or ad spend.
* The viral threshold uses raw relative VPD (not age-adjusted), and viral videos cluster around a small number of releases, so treat the rates as patterns, not proof.

In [147]:
# Analysis Monthly Trend By Video Type
monthly_trend = (
    yt_eval.groupby(["month", "video_type"], observed=False)
    .agg(
        n_videos=("video_id", "count"),
        median_vpd=("views_per_day", "median"),
        median_rel_vpd_adj=("rel_vpd_age_adj", "median"),
    )
    .round(2)
    .unstack("video_type")
)

print("\n"+"="*70)
print("---------------------------- MONTHLY TREND --------------------------- ")
print("="*70)
print(monthly_trend.to_string())

# Year-over-year (2026 is partial and its videos are younger)
yoy_summary = (
    yt_eval.groupby(["year", "video_type"], observed=False)
    .agg(
        n_videos=("video_id", "count"),
        median_vpd=("views_per_day", "median"),
        median_rel_vpd_adj=("rel_vpd_age_adj", "median"),
        median_eng=("engagement_pct", "median"),
    )
    .round(2)
)
print("\n"+"="*70)
print("------------------------ YEAR-OVER-YEAR SHIFT ------------------------ ")
print("="*70)
print(yoy_summary)


---------------------------- MONTHLY TREND --------------------------- 
           n_videos        median_vpd          median_rel_vpd_adj       
video_type     Long Shorts       Long   Shorts               Long Shorts
month                                                                   
2025-01          32     91     450.17   266.46               1.16   0.68
2025-02          42     99     489.07   353.48               0.94   0.65
2025-03          49    131     570.27   390.67               1.00   0.66
2025-04          55    131     630.72   631.02               1.03   1.10
2025-05          46    169     571.86   600.02               1.03   1.15
2025-06          52    208     515.07   835.65               0.87   1.16
2025-07          81    297     550.22   612.62               1.06   1.06
2025-08          56    164     710.02   822.78               0.92   0.80
2025-09          58    202     548.73 1,179.59               0.83   0.82
2025-10          62    166     623.32   802.15     

In [148]:
# Only videos inside a comeback window (break videos do not belong to a release)
in_window = yt_eval[yt_eval["phase"] != "break"].dropna(subset=["release_id"])
vid_rel = in_window.merge(
    rel_main[["release_id", "release_name", "release_date", "type"]].drop_duplicates(),
    on="release_id", how="left",
)

# Identifying Which Release had the Best-Performing Content
release_bench = (
    vid_rel.groupby(["group", "release_date", "release_name", "type"], observed=True)
    .agg(
        n_videos=("video_id", "count"),
        shorts_pct=("video_type", lambda x: (x == "Shorts").mean() * 100),
        median_rel_vpd=("relative_vpd", "median"),
        median_rel_vpd_adj=("rel_vpd_age_adj", "median"),
        median_eng=("engagement_pct", "median"),
    )
    .round(2)
    .reset_index()
)
release_bench["small_n"] = np.where(release_bench["n_videos"] < 30, "*", "")

print("\n"+"="*80)
print("--------------------------- RELEASE-LEVEL BENCHMARK ---------------------------- ")
print("="*80)
print(release_bench.sort_values(["group", "median_rel_vpd_adj"],
                                ascending=[True, False]).to_string(index=False))
print("\n* = fewer than 30 videos in the release window")


--------------------------- RELEASE-LEVEL BENCHMARK ---------------------------- 
        group release_date                                  release_name           type  n_videos  shorts_pct  median_rel_vpd  median_rel_vpd_adj  median_eng small_n
        aespa   2026-05-11 WDA (Whole Different Animal) (feat. G-DRAGON) digital single        83       85.54            1.10                1.03        6.13        
        aespa   2026-05-29                                      LEMONADE   studio album       162       85.19            1.07                1.00        6.08        
        aespa   2025-06-27                                    Dirty Work   single album       116       82.76            0.58                0.99        5.70        
        aespa   2026-07-24                                   KISS N TELL     mini album       163       78.53            1.73                0.84        6.49        
        aespa   2025-09-05                                      Rich Man     mini album

In [149]:
cols_show = ["group", "video_type", "phase", "title", "views",
             "views_per_day", "relative_vpd", "engagement_pct"]
pd.set_option("display.max_colwidth", 50)

print("\n"+"="*95)
print("--------------------------- TOP 3 PER GROUP (HIGHEST RELATIVE VPD) ---------------------------- ")
print("="*95)

top_per_group = (
    yt_eval.sort_values("relative_vpd", ascending=False)
    .groupby("group", observed=True).head(3)
    .sort_values(["group", "relative_vpd"], ascending=[True, False])
)
print(top_per_group[cols_show].to_string(index=False))


print("\n"+"="*95)
print("--------------------------- BOTTOM 3 PER GROUP (LOWEST RELATIVE VPD) -------------------------- ")
print("="*95)
bottom_per_group = (
    yt_eval.sort_values("relative_vpd", ascending=True)
    .groupby("group", observed=True).head(3)
    .sort_values(["group", "relative_vpd"], ascending=[True, True])
)
print(bottom_per_group[cols_show].to_string(index=False))


--------------------------- TOP 3 PER GROUP (HIGHEST RELATIVE VPD) ---------------------------- 
        group video_type           phase                                                                         title     views  views_per_day  relative_vpd  engagement_pct
        aespa       Long           break                                                               왜요 제가 좀 신나 보여요?   1443831     206,261.57        224.58            1.44
        aespa     Shorts during_comeback      that’s Dirty Work! Show us your  moves! Brought to you by YouTube Shorts 167860560     326,576.96        125.83            0.74
        aespa     Shorts during_comeback Sharp eyes, fierce lookShow off your  moves. Brought to you by YouTube Shorts 145090537     283,379.96        109.19            0.77
hearts2hearts     Shorts           break                                                                      친구 태그 ㄱㄱ  59785918     649,846.93        184.69            1.49
hearts2hearts     Shorts        

In [150]:
# Identyfying how concentrated are views
sorted_views = yt_eval["views"].sort_values(ascending=False)
total = sorted_views.sum()
for label, k in [("Top 10 videos", 10),
                 ("Top 1%", int(len(yt_eval) * 0.01)),
                 ("Top 10%", int(len(yt_eval) * 0.10))]:
    print(f"{label}: {sorted_views.head(k).sum() / total * 100:.1f}% of all views")

conc = yt_eval.groupby("group")["views"].apply(
    lambda s: s.nlargest(max(1, int(len(s) * 0.1))).sum() / s.sum() * 100
).round(1)
print("\nShare of views from each group's top 10% videos:\n", conc)

# Looking where do viral videos (relative VPD > 10) come from?
viral = yt_eval[yt_eval["relative_vpd"] > 10]
print(f"\n{len(viral)} videos with relative VPD > 10")
print(viral.groupby(["video_type", "phase"], observed=True).size().unstack(fill_value=0))
print(viral["group"].value_counts())

Top 10 videos: 17.2% of all views
Top 1%: 30.2% of all views
Top 10%: 61.2% of all views

Share of views from each group's top 10% videos:
 group
aespa           68.60
hearts2hearts   51.10
nct 127         45.30
nct dream       38.40
nct wish        54.30
riize           53.30
wayv            61.40
Name: views, dtype: float64

310 videos with relative VPD > 10
phase       pre_comeback  during_comeback  post_comeback  break
video_type                                                     
Long                  15               17             18     24
Shorts                18               95             82     41
group
hearts2hearts    98
riize            61
aespa            55
nct wish         40
nct dream        25
wayv             25
nct 127           6
Name: count, dtype: int64


In [151]:
# Viral rate (relative VPD > 10) per phase and format
yt_eval["is_viral"] = yt_eval["relative_vpd"] > 10
print((yt_eval.groupby(["video_type", "phase"], observed=True)["is_viral"].mean() * 100).round(1).unstack("video_type"))

# Posts per month, 2025 vs 2026 (2026 has 11 months)
print((yt_eval.groupby(["year", "video_type"]).size().unstack().div([12, 11], axis=0)).round(0))

video_type       Long  Shorts
phase                        
pre_comeback     6.50    1.90
during_comeback  7.00    6.80
post_comeback    6.10    7.50
break            4.50    3.10
video_type  Long  Shorts
year                    
2025       55.00  166.00
2026       59.00  254.00


### 6. Data Modeling: Star Schema Export for BI Dashboard
To prepare the cleaned dataset for interactive visualization (e.g., Power BI / Tableau), the evaluated data is structured into a **Star Schema** consisting of one central fact table and two dimension tables:

* **`fact_videos` (Fact Table):** Contains video-level quantitative metrics (`views`, `views_per_day`, `relative_vpd`, `rel_vpd_age_adj`, `engagement_pct`), temporal attributes (KST), and binary flags (`is_viral`).
* **`dim_release` (Dimension Table):** Stores metadata for the 28 main comeback releases, enriched with `release_tier` (`album` vs. `digital single`).
* **`dim_group` (Dimension Table):** Provides standardized artist labels (`group_label`) and strategic groupings (`is_nct`, `is_core`).

**Notes**:
- Redundant attributes (such as `group` inside `dim_release`) are removed to prevent ambiguous/circular relationships in BI tools.
- Exported with `utf-8-sig` encoding to preserve Korean Hangul characters and emojis, followed by primary/foreign key integrity checks.


In [152]:
# print(yt_eval.columns.tolist)
# print(rel_main.columns.tolist)

# Split Data into Star Schema (Fact & Dimension Tables) for BI Dashboard
OUT = PROCESSED / "dashboard"
OUT.mkdir(parents=True, exist_ok=True)


# Fact Table
fact_cols = [
    "video_id", "group", "release_id", "video_type", "phase", "time_slot",
    "day_kst", "hour_kst", "title", "duration", "views", "likes", "comments",
    "comments_disabled", "age_days", "views_per_day", "engagement_pct",
    "relative_vpd", "rel_vpd_age_adj", "relative_engagement",
    "days_from_release", "is_viral",
]
fact_videos = yt_eval[fact_cols].copy()
fact_videos["published_date"] = (yt_eval["published_at"] + pd.Timedelta(hours=9)).dt.date
for col in ["phase", "time_slot", "day_kst"]:
    fact_videos[col] = fact_videos[col].astype(str)

# Release Dimension (removed 'group' to prevent circular joins in BI tools)
dim_release = rel_main[["release_id", "release_name", "release_date", "type"]].drop_duplicates().copy()
dim_release["release_tier"] = np.where(
    dim_release["type"].isin(["single album", "mini album", "studio album"]),
    "album", "digital single",
)

# Group Dimension
dim_group = pd.DataFrame({"group": sorted(yt_eval["group"].unique())})
dim_group["group_label"] = dim_group["group"].str.upper().str.replace("HEARTS2HEARTS", "H2H")
dim_group["is_nct"] = dim_group["group"].str.startswith(("nct", "wayv"))
dim_group["is_core"] = dim_group["group"].isin(["aespa", "riize", "hearts2hearts"])

# Export to CSV (utf-8-sig preserves Korean Hangul & emojis)
tables = {"fact_videos": fact_videos, "dim_release": dim_release, "dim_group": dim_group}
for name, d in tables.items():
    d.to_csv(OUT / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name}: {d.shape}")

# Referential Integrity Checks
print("\n========= DATA INTEGRITY CHECKS =========")
print("Duplicate video_id:", fact_videos["video_id"].duplicated().sum())
print("Groups missing in dim_group:", set(fact_videos["group"]) - set(dim_group["group"]))
print("release_id missing in dim_release:", set(fact_videos["release_id"].dropna()) - set(dim_release["release_id"]))


fact_videos: (6086, 23)
dim_release: (28, 5)
dim_group: (7, 4)

========= DATA INTEGRITY CHECKS =========
Duplicate video_id: 0
Groups missing in dim_group: set()
release_id missing in dim_release: set()


## Conclusion

### Key findings
1. **Shorts dominate uploads** (78.6% of videos), and their volume grew from ~166 to ~254 per month in 2026, while Long stayed flat at ~55-59 per month.
2. **Long-form videos benefit most from comeback momentum**: median relative VPD is ~1.3x baseline in the pre and during phases, then ~0.9x after. The pattern holds after age adjustment.
3. **Comebacks change the tail of Shorts, not the typical Short.** Viral Shorts make up ~7% of uploads during and after a comeback, vs 1.9% before and 3.1% in break, while median performance stays near 1.0x.
4. **Upload timing matters little.** The only clear pattern is that Shorts posted at night or in the morning (KST) underperform by ~25%.
5. **Views are highly concentrated**: the top 10% of videos hold 61% of all views, and the top 10 videos alone hold 17%.
6. **Engagement per viewer is flat** across phases, days, and time slots, so comebacks and timing affect reach, not interaction.

### Recommendations for the marketing team
* **Plan key Long-form content (MVs, dance practices, performance videos) for the weeks before and during a release**, when Long videos perform ~30% above baseline.
* **Keep Shorts steady year-round.** Their typical performance does not depend on the comeback phase, so they suit regular calendar filling.
* **Avoid posting Shorts at night or in the morning (KST).** Any time between 12:00 and 23:59 performs similarly.
* **Treat flagship content as the main reach driver.** Because views are concentrated in a few videos, these deserve the most planning, and a single viral video should not be used as the benchmark for typical performance.
* **Test other upload hours for Long videos.** 94% are posted in the evening, so the data cannot say whether other times would work better.

### Limitations to keep in mind
* Results are descriptive patterns from YouTube only (2025-01 to 2026-11), not causal proof.
* Videos cluster around 28 releases, so the effective sample is much smaller than 6,086 videos.
* Content type is not controlled, so timing and phase effects may partly reflect what was posted.
* Phase windows and the inclusion of digital singles as main releases are assumptions, and no sensitivity check was run.